# Retrieval

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors.listwise_rerank import LLMListwiseRerank

from dotenv import load_dotenv

load_dotenv()

## Vector store

In [ ]:
client = QdrantClient(url="http://localhost:6333")

In [ ]:
embeddings = HuggingFaceEmbeddings(model="sdadas/st-polish-paraphrase-from-distilroberta")

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="knowledge_database",
    embedding=embeddings,
)

In [ ]:
query = "Zwierzęta domowe"

# Gra w warcaby
# Kto ma zająca?
# Wszystko o Bartku
# Kto czym jeździ?

In [ ]:
results = vector_store.similarity_search_with_score(query, k=4)

In [ ]:
for document, score in results:
    print("Score:", score)
    print("Document content:", document.page_content)
    print()

## Retriever

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [ ]:
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
template = """Retrieved documents:
---------
{context}

-----
End of retrieved documents
============

Answer the following question: {input}.

=============

Important information: You can answer only based on the context from the retrieved documents.
If you don't have any information regarding the question in the context, say 'I don't know'"""


prompt_template = PromptTemplate.from_template(template)

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
query = "Powiedz mi kto czym jeździ"
response = retrieval_chain.invoke({"input": query})

In [ ]:
response

In [ ]:
response["input"]

In [ ]:
response["answer"]

In [ ]:
context_docs = [doc.page_content for doc in response["context"]]
context_docs

**Finalny prompt**

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler


class PromptCapture(BaseCallbackHandler):
    def __init__(self):
        self.prompt = ""
    
    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.prompt = "\n\n".join(
            f"[{msg.type.upper()}]: {msg.content}" 
            for msg_list in messages 
            for msg in msg_list
        )

In [ ]:
capture = PromptCapture()
response = retrieval_chain.invoke({"input": query}, config={"callbacks": [capture]})

In [ ]:
response["answer"]

In [ ]:
print(capture.prompt)

---

In [ ]:
query = "Podaj 3 przykłady owoców"
response = retrieval_chain.invoke({"input": query})

In [ ]:
response["answer"]

**Filtrowanie payloadu**

Filtrowanie payloadu odbywa się w oparciu o trzy możliwe argumenty przekazywane do `vector_store.as_retriever()`:
- `must` (lista warunków połączonych spójnikiem `and`)
- `should` (lista warunków połączonych spójnikiem `or`)
- `must_not` (lista warunków które nie mogą być spełnione)

Każdy z powyższych parametrów przyjmuje listę obiektów `models.FieldCondition()`, które to z kolei obiekty przyjmują dwa argumenty:
- `key` - klucz payloadu, po którym filtrujemy
- `match` - rodzaj dopasowania, np. `models.MatchValue()` (dokładna wartość), `models.MatchAny()` (dopasowanie do dowolnej wartości z podanej listy), `models.MatchText()` (wyszukiwanie w tekście)

In [ ]:
from qdrant_client import models

retriever = vector_store.as_retriever(search_kwargs={
        "k": 2,
        "filter": models.Filter(
            must=[
                models.FieldCondition(
                    key="metadata.person",
                    match=models.MatchValue(value="Andrzej") 
                )
            ]
        )
    }
)

In [ ]:
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
query = "Powiedz mi kto czym jeździ"
response = retrieval_chain.invoke({"input": query})
response["answer"]

In [ ]:
[item.page_content for item in response["context"]]

> **ZADANIA**

## Reranking

- Celem rerankingu jest zmiana kolejności dokumentów pozyskanych z bazy wektorowej w prompcie
- Może się to odbywać na różne sposoby, najczęściej wysyłamy wszystkie dokumenty do LLMa a on "świadomie" zmienia ich kolejność zgodnie z tym, które dokumenty uważa za najbardziej pasujące do zadanego pytania
- Podobieństwo wektorowe traktujemy więc jako sito ale faktyczny ranking wykonuje teraz LLM
- Oprócz zmiany kolejności istnieje też możliwość wzięcia tylko *n* najbardziej pasujących dokumentów. Dzięki temu oszczędzamy zużycie tokenów bo finalnie do LLMa trafi mniej dokumentów, ale (w teorii) tylko te najbardziej odpowiednie

Reranking nie jest niezbędny, ale warto go robić ponieważ badania pokazują, że LLMy większą uwagę przykładają do tego co jest na początku i końcu kontekstu a na środek nie zwracają aż takiej uwagi. Dlatego warto najpierw podać najważniejsze dokumenty, potem pozostałe a na końcu zadać pytanie.

In [ ]:
reranker = LLMListwiseRerank.from_llm(llm=llm, top_n=4)

**Przykład 1**

In [ ]:
query = "Witaj"
docs = [
    "Pa pa",
    "Mówić",
    "Cześć",
    "Pozdrowienie"
]

In [ ]:
from langchain_core.documents import Document

In [ ]:
doc_objects = [Document(page_content=text) for text in docs]

reranked_docs = reranker.compress_documents(doc_objects, query)

results = [doc.page_content for doc in reranked_docs]
results

**Przykład 2**

In [ ]:
query = "Jakie są korzyści zdrowotne zielonej herbaty?"

docs = [
    "Czarna herbata ma intensywniejszy smak niż zielona herbata.",
    "Zielona herbata zawiera antyoksydanty, które mogą wspomagać zdrowie serca.",
    "Zielona herbata może pomóc w odchudzaniu i poprawie funkcji mózgu.",
    "Kawa ma więcej kofeiny niż zielona herbata.",
    "Herbaty ziołowe różnią się od zielonej herbaty pod względem składu i działania."
]

In [ ]:
doc_objects = [Document(page_content=text) for text in docs]

reranked_docs = reranker.compress_documents(doc_objects, query)

results = [doc.page_content for doc in reranked_docs]
results

### Integracja z retrieverem

**Wersja bez rerankingu**

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name="knowledge_database",
    embedding=embeddings,
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o")

In [ ]:
template = """Retrieved documents:
---------
{context}

-----
End of retrieved documents
============

Answer the following question: {input}.

=============

Important information: You can answer only based on the context from the retrieved documents.
If you don't have any information regarding the question in the context, say 'I don't know'"""


prompt_template = PromptTemplate.from_template(template)

In [ ]:
document_chain = create_stuff_documents_chain(llm, prompt_template)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
query = "Kto co robi?"
response = retrieval_chain.invoke({"input": query})

In [ ]:
response["answer"]

In [ ]:
retrieved_docs = response["context"]
[doc.page_content for doc in retrieved_docs]

**Wersja z rerankingiem**

In [ ]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 4})
reranker = LLMListwiseRerank.from_llm(llm=llm, top_n=3)

In [ ]:
retriever_with_reranker = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=base_retriever
)

In [ ]:
retrieval_chain = create_retrieval_chain(retriever_with_reranker, document_chain)

In [ ]:
query

In [ ]:
response = retrieval_chain.invoke({"input": query})

In [ ]:
response["input"]

In [ ]:
response["answer"]

In [ ]:
context_docs = [doc.page_content for doc in response["context"]]
context_docs